In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_API_KEY=os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT=os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION=os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_MODEL=os.getenv("AZURE_OPENAI_MODEL")
AZURE_SEARCH_ENDPOINT=os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY=os.getenv("AZURE_SEARCH_API_KEY")
AZURE_SEARCH_INDEX_NAME=os.getenv("AZURE_SEARCH_INDEX_NAME")

for name, value in {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    "AZURE_OPENAI_MODEL": AZURE_OPENAI_MODEL,
    "AZURE_SEARCH_ENDPOINT": AZURE_SEARCH_ENDPOINT,
    "AZURE_SEARCH_API_KEY": AZURE_SEARCH_API_KEY,
    "AZURE_SEARCH_INDEX_NAME": AZURE_SEARCH_INDEX_NAME
}.items():
    print(f"{name:30s} -> {'OK' if value else 'MISSING'}")

AZURE_OPENAI_API_KEY           -> OK
AZURE_OPENAI_ENDPOINT          -> OK
AZURE_OPENAI_API_VERSION       -> OK
AZURE_CHAT_SEARCH              -> OK
AZURE_CHAT_CLASSIFIER          -> OK


In [4]:
import pandas as pd

df = pd.read_csv("tickets_IT_helpdesk.csv")

df.head()

,ticket-id,issue,category,expected_resolution
0,TKT-001,Karyawan tidak bisa login Windows karena lupa ...,Access,Arahkan ke self-service portal https://passwor...
1,TKT-002,Akun domain Windows terkunci setelah salah mem...,Access,Verifikasi NIK karyawan lalu unlock akun via c...
2,TKT-003,Token MFA tidak berfungsi setelah karyawan men...,Access,Panduan reset MFA mandiri di https://my.bsi.mi...
3,TKT-004,QR Code MFA tidak muncul saat proses pendaftar...,Access,Pastikan karyawan menggunakan browser laptop (...
4,TKT-005,Karyawan tidak bisa konek GlobalProtect VPN sa...,Network,Lakukan Sign Out → restart laptop → login ulan...


1. Bikin ask_query
2. Bikin classify_query (Access, Network, Hardware, Software)
3. Bikin response_generate
4. Result = query, response, context di save di json

In [ ]:
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_API_KEY)
)


In [ ]:
VALID_CATEGORIES = {"Access", "Network", "Hardware", "Software"}

def build_classification(top_k: int = 3) -> str:
    lines = []
    for category in VALID_CATEGORIES:
        samples = df[df["category"] == category].head(top_k)
        for _, row in samples.iterrows():
            lines.append(f"issue: '{row["issue"]}'\nCategory: {category}")
    return "\n\n".join(lines)

def classify_query(query: str) -> str:
    few_shot = build_classification(top_k=2)

    system_prompt = (
        "You are an IT helpdesk ticket classifier. "
        "Below are example tickets and their correct categories. "
        "Learn from these examples, then classify the new issue>\n\n"
        "== Example Tickets ==\n"
        f"{few_shot}\n"
        "== End Examples ==\n"
        "Valid categories: Access, Network, Hardware, Software\n"
        "If the issue do not fit any of these categories, respond with exactly:"
        "I don't have that information\n"
        "Otherwise, respond with exactly ONE word from the valid categories. "
        "Do NOT include any other text, punctuation, or explanation."
    )

    response = client.chat.completions.create(
        model=AZURE_OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0,
        max_tokens=10
    )

    raw=response.choices[0].message.content.strip()

    category = raw.capitalize()

    return category

print(classify_query("Bagaimana cara mengaktifkan MFA?"))
print(classify_query("Cara membuat nasi goreng"))

In [ ]:
def response_generate(query: str, category: str) -> tuple[str, list[dict]]:
    if category not in VALID_CATEGORIES:
        return "I don't have that information.", []

    results = search_client.search(
        search_text=query,
        top=3
    )

    system_prompt = (
        "You are a helpful IT helpdesk agent. "
        "Use the reference tickets below to answer the user's question. "
        "Be concise and actionable.\n\n"
    )

    response = client.chat.completions.create(
        model=AZURE_OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ],
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": AZURE_SEARCH_ENDPOINT,
                        "index_name": AZURE_SEARCH_INDEX_NAME,
                        "authentication": {
                            "type": "api_key",
                            "key": AZURE_SEARCH_API_KEY
                        },
                        "filter": f"category eq '{category}'"
                    }
                }
            ]
        },
        temperature=0.3,
        max_tokens=512
    )

    answer = response.choices[0].message.content.strip()
    context = response.choices[0].message.content

    context_text = "\n".join([c.content for c in context.citations]) if context and context.citations else ""


    return {
        "query": query,
        "response": answer,
        "context": context_text
    }

In [ ]:
import jsonlines

results = []

for _, row in df.iterrows():
    query = row["issue"]
    category = classify_query(query)
    if category not in VALID_CATEGORIES:
        pass
    else:
        result = response_generate(query, category)
        results.append(result)
        print(f"Done: {row['ticket-id']} | category: {category}")

with jsonlines.open("results.jsonl", mode="w") as writer:
    writer.write_all(results)

print(f"Saved {len(results)} records to results.jsonl")